In [ ]:
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from optuna.trial import TrialState
from optuna.visualization import plot_optimization_history, plot_param_importances

In [ ]:
# Wczytanie zbioru danych
X, y = load_breast_cancer(return_X_y=True)

# Podzial na czesc treningowa (80%) i testowa (20%)
# Argument stratify zapewnia zachowanie proporcji klas w obu czesciach
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model bazowy z domyslnymi hiperparametrami
baseline = RandomForestClassifier(random_state=42)

# Ocena modelu bazowego metoda pieciokrotnej walidacji krzyzowej
baseline_cv = cross_val_score(baseline, X_train, y_train, cv=5, scoring="accuracy")
print(
    f"Baseline, dokladnosc CV: {baseline_cv.mean():.4f} (odch. std. {baseline_cv.std():.4f})"
)

# Trening na calym zbiorze treningowym i jednokrotna ocena na zbiorze testowym
baseline.fit(X_train, y_train)
baseline_test_acc = baseline.score(X_test, y_test)
print(f"Baseline, dokladnosc na zbiorze testowym: {baseline_test_acc:.4f}")

In [ ]:
def objective(trial):
    # Definicja przestrzeni przeszukiwania w paradygmacie define-by-run.
    # Kazde wywolanie metody suggest deklaruje jeden hiperparametr i jego zakres.
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        # Nowe parametry rozszerzające
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
    }
    # Budowa modelu z zaproponowana konfiguracja
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)

    # Ocena konfiguracji metoda pieciokrotnej walidacji krzyzowej
    score = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy").mean()
    # Zwracana wartosc bedzie maksymalizowana przez Optune
    return score

In [ ]:
def objective_przeglad(trial):
    # Liczba calkowita z przedzialu domknietego
    n_estimators = trial.suggest_int("n_estimators", 50, 400)

    # Liczba calkowita z zadanym krokiem
    max_depth = trial.suggest_int("max_depth", 2, 20, step=2)

    # Liczba zmiennoprzecinkowa z przedzialu domknietego
    subsample = trial.suggest_float("subsample", 0.5, 1.0)

    # Liczba zmiennoprzecinkowa probkowana w skali logarytmicznej
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)

    # Wybor kategoryczny z listy wartosci
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])

#### Optuna

In [ ]:
# Utworzenie badania. Dokladnosc jest maksymalizowana.
study = optuna.create_study(direction="maximize", study_name="rf_tuning")

# Uruchomienie optymalizacji obejmujacej 50 prob
study.optimize(objective, n_trials=50)

print("Najlepsza wartosc funkcji celu (dokladnosc CV):", study.best_value)
print("Najlepsza konfiguracja hiperparametrow:", study.best_params)

best_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)
tuned_test_acc = best_model.score(X_test, y_test)

print(f"Model strojony, dokladnosc na zbiorze testowym:{tuned_test_acc:.4f}")
print(f"Model bazowy, dokladnosc na zbiorze testowym:{baseline_test_acc:.4f}")

#### Pruner

In [ ]:
# Wydzielenie zbioru walidacyjnego z czesci treningowej
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

def objective_pruning(trial):
    params = {
        # Celowo duza, stala liczba etapow. O momencie przerwania decyduje pruner.
        "n_estimators": 300,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
    }

    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(X_tr, y_tr)

    # Ocena modelu etapami, po kolejnych drzewach
    for step, y_pred in enumerate(model.staged_predict(X_val)):
        acc = accuracy_score(y_val, y_pred)

        # Raportowanie wyniku posredniego
        trial.report(acc, step)

        # Sprawdzenie decyzji prunera i ewentualne przerwanie proby
        if trial.should_prune():
            raise optuna.TrialPruned()

    return acc  # type: ignore

study_pruned = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,  # pierwsze piec prob nigdy nie jest przycinanych
        n_warmup_steps=20,  # zadna proba nie jest przycinana przed dwudziestym krokiem
    ),
)
study_pruned.optimize(objective_pruning, n_trials=40)  # type: ignore

# Zestawienie liczby prob ukonczonych i przycietych
pruned = [t for t in study_pruned.trials if t.state == TrialState.PRUNED]
complete = [t for t in study_pruned.trials if t.state == TrialState.COMPLETE]
print(f"Proby ukonczone: {len(complete)}, proby przyciete:{len(pruned)}")

#### Wizualizacja

In [ ]:
# Historia optymalizacji
fig1 = plot_optimization_history(study)
fig1.show()

# Waznosc hiperparametrow
fig2 = plot_param_importances(study)
fig2.show()